# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [13]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [26]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    # Not a plain `git pull` -- if this clone has ANY local changes (e.g. leftover
    # checkpoints/outputs from an earlier run in the same runtime that never got pushed),
    # a pull can fail outright ("local changes would be overwritten") and Colab just
    # prints the error and moves on -- training then silently proceeds on stale code with
    # no visible failure until much later (e.g. a non-fast-forward push at the end).
    # fetch + hard reset guarantees this checkout exactly matches origin/{BRANCH} no
    # matter what state it was left in.
    !cd ECE1508_GenAI && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}

%cd ECE1508_GenAI
!git log --oneline -1


remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 107 (delta 55), reused 89 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 2.77 MiB | 17.29 MiB/s, done.
Resolving deltas: 100% (55/55), completed with 10 local objects.
From https://github.com/WoodyChang21/ECE1508_GenAI
 * branch            steven     -> FETCH_HEAD
   69f6c64..ea38d52  steven     -> origin/steven
HEAD is now at ea38d52 Trade-confidence redesign: per-model quality gates + per-model return thresholds
/content/ECE1508_GenAI/ECE1508_GenAI/ECE1508_GenAI
ea38d52 (HEAD -> steven, origin/steven) Trade-confidence redesign: per-model quality gates + per-model return thresholds


In [15]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

In [16]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [17]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI/ECE1508_GenAI
plugins: anyio-4.14.2, typeguard-4.5.2, langsmith-0.10.2
collected 33 items                                                             

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  3%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [  6%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [  9%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 12%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 15%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 18%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [18]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

22:11:45 device: cuda
22:11:45 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:11:45 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:11:49 epoch 1/20  train_loss=0.23157  val_loss=0.10953  (2.4s)
22:11:49   -> saved best checkpoint (val_loss=0.10953) to steven/outputs/patchtst_checkpoint.pt
22:11:51 epoch 2/20  train_loss=0.16731  val_loss=0.11391  (1.9s)
22:11:53 epoch 3/20  train_loss=0.15137  val_loss=0.09968  (1.9s)
22:11:53   -> saved best checkpoint (val_loss=0.09968) to steven/outputs/patchtst_checkpoint.pt
22:11:55 epoch 4/20  train_loss=0.14287  val_loss=0.09491  (2.0s)
22:11:55   -> saved best checkpoint (val_loss=0.09491) to steven/outputs/patchtst_checkpoint.pt
22:11:57 epoch 5/20  train_loss=0.14188  val_loss=0.09595  (1.9s)
22:11:59 epoch 6/20  train_loss=0.13820  val_loss=0.09251  (1.9s)
22:11:59   -> saved best checkpoint (val_loss=0.09251) to steven/outputs/patchtst_checkpoint.pt
22:12:01 epoch 7/20  train_

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [19]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

22:12:28 device: cuda
22:12:29 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:12:29 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:12:32 epoch 1/30  beta=0.20  train_recon=0.43672 (kl=1.2021)  val_recon=0.20135 (kl=1.2000)  (2.0s)
22:12:32   -> saved best checkpoint (val_recon=0.20135) to steven/outputs/cvae_checkpoint.pt
22:12:33 epoch 2/30  beta=0.40  train_recon=0.30095 (kl=1.2000)  val_recon=0.19546 (kl=1.2000)  (1.4s)
22:12:33   -> saved best checkpoint (val_recon=0.19546) to steven/outputs/cvae_checkpoint.pt
22:12:35 epoch 3/30  beta=0.60  train_recon=0.27071 (kl=1.2000)  val_recon=0.16172 (kl=1.2000)  (1.4s)
22:12:35   -> saved best checkpoint (val_recon=0.16172) to steven/outputs/cvae_checkpoint.pt
22:12:36 epoch 4/30  beta=0.80  train_recon=0.22549 (kl=1.2002)  val_recon=0.15575 (kl=1.2000)  (1.4s)
22:12:36   -> saved best checkpoint (val_recon=0.15575) to steven/outputs/cvae_checkpoint.pt
22:12:38 epoch 5/30  be

## Evaluate both models on the fixed test set

In [20]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

22:13:16 device: cuda
22:13:17 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:13:17 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:13:17 sell-price shrink bound: p99.0 of |anchored log return| over train = 0.0190 (vs. model's own MAX_LOG_RETURN)
22:13:17 running walk-forward backtest (ctx=70 bars, confidence>=0.50, min_return>=0.100%, 24537..27006)...
22:13:29 walk-forward: PatchTST 339 trades / 1719 decisions, CVAE 0 trades / 2397 decisions
22:13:29 wrote metrics to steven/outputs/metrics.json
22:13:29 walk_forward: {
  "ctx_bars": 70,
  "confidence_threshold": 0.5,
  "min_return_threshold": 0.001,
  "buy_and_hold": {
    "entry_date": "2024-01-16",
    "entry_price": 474.95,
    "exit_date": "2025-05-30",
    "exit_price": 589.46,
    "elapsed_years": 1.3689253935660506,
    "total_return": 0.24109906305926954,
    "annual_return": 0.17091564847596086
  },
  "naive_periodic": {
    "n_trades": 799,
    "total_return": -

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [21]:
!python steven/src/update_report.py

22:13:31 updated steven/v1.md: results-samples, hit-summary, spread-summary, buy-hold-benchmark, walk-forward-strategies, walk-forward-outcome-breakdown
22:13:31 not auto-updated -- reread and edit by hand if the story changed: the 'In plain terms' / 'A subtle but important point' interpretation paragraphs under Results, the 'pre-retrain checkpoints' caveat in Results, and the 'Retrain both models' checkbox under Next steps.


## Sync results back to GitHub

Commits `steven/outputs/` (checkpoints, metrics.json, sample_plots) and the regenerated `steven/v1.md` from this Colab runtime and pushes straight to the `steven` branch -- no manual zip/download step. That step wasn't reliably reaching the local machine: `files.download()`'s browser-download trick only works from the Colab web UI, not when this kernel is attached remotely (e.g. from VS Code's kernel picker), so nothing ever landed on disk.

Needs a GitHub personal access token with `repo` write scope for this push only -- entered via `getpass` below, never written to the notebook or committed anywhere.

In [22]:
# %%bash
# git fetch origin steven
# git merge origin/steven --no-edit

In [23]:
import getpass

token = getpass.getpass("GitHub PAT (repo write, used only for this push): ")

In [24]:
%%bash -s "$token"
TOKEN="$1"
if [ -z "$TOKEN" ]; then
  echo "Token was empty -- re-run the getpass cell above and actually paste your PAT before pressing Enter." >&2
  exit 1
fi
git config user.email "colab@ephemeral.local"
git config user.name "Colab Runtime"
git add steven/outputs steven/v1.md
if git diff --cached --quiet; then
  echo "Nothing new to commit -- outputs/v1.md unchanged from last commit."
else
  git commit -m "Retrain + refresh results from Colab run"
fi
# Push unconditionally -- a prior run may have committed but failed to push (e.g. a blank
# token), in which case there's nothing new to commit here but HEAD is still ahead of origin.
git push "https://${TOKEN}@github.com/WoodyChang21/ECE1508_GenAI.git" HEAD:steven

[steven c6fdf18] Retrain + refresh results from Colab run
 11 files changed, 208 insertions(+), 508 deletions(-)
 rewrite steven/outputs/cvae_checkpoint.pt (92%)
 delete mode 100644 steven/outputs/sample_plots/lose_expiry_start26890_ctx70.png
 delete mode 100644 steven/outputs/sample_plots/no_trade_start25344_ctx70.png
 create mode 100644 steven/outputs/sample_plots/no_trade_start25786_ctx70.png
 rewrite steven/outputs/sample_plots/samples.json (85%)
 create mode 100644 steven/outputs/sample_plots/skipped_start25006_ctx70.png
 delete mode 100644 steven/outputs/sample_plots/skipped_start26439_ctx70.png
 delete mode 100644 steven/outputs/sample_plots/win_expiry_start24903_ctx70.png
 delete mode 100644 steven/outputs/sample_plots/win_take_profit_start26752_ctx70.png


To https://github.com/WoodyChang21/ECE1508_GenAI.git
   4465c5d..c6fdf18  HEAD -> steven


### Fallback: zip + browser download

Only useful if you're running this notebook inside the actual Colab web UI (not a remote kernel) and would rather download a zip than push through git.

In [25]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

updating: steven/outputs/ (stored 0%)
updating: steven/outputs/metrics.json (deflated 67%)
updating: steven/outputs/cvae_checkpoint.pt (deflated 7%)
updating: steven/outputs/sample_plots/ (stored 0%)
updating: steven/outputs/sample_plots/samples.json (deflated 75%)
updating: steven/outputs/patchtst_checkpoint.pt (deflated 9%)
  adding: steven/outputs/sample_plots/skipped_start25006_ctx70.png (deflated 11%)
  adding: steven/outputs/sample_plots/no_trade_start25786_ctx70.png (deflated 12%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>